In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#r "nuget: FsLexYacc.Runtime, 11.3.0"
#r "GrammarParser/bin/Debug/net9.0/GrammarParser.dll"
open System
open System.IO
open System.Net
open LectureGrammar

# Converting a Grammar into <span style="font-variant:small-caps;">Html</span>

Store the lexical specification in `GrammarParser/Lexer.fsl` and the grammar of grammars in `GrammarParser/Parser.fsy`. This F# version completes the original unfinished ANTLR exercise with FsLexYacc. Variables and quoted literals become typed `Item` cases; a rule records its head and alternatives.

In [ ]:
File.ReadAllText("GrammarParser/Parser.fsy") |> printfn "%s"

In [ ]:
File.ReadAllText("GrammarParser/Lexer.fsl") |> printfn "%s"

The file `c-grammar.g` contains a context-free grammar for the language `C`.

In [ ]:
let source = File.ReadAllText("c-grammar.g")
printfn "%s" source

Our goal is to convert this grammar into an <span style="font-variant:small-caps;">Html</span> <a href="c-grammar.html">file</a>.

The adjacent `.fsproj` generates and compiles the scanner and parser at build time.

In [ ]:
// Build GrammarParser/GrammarParser.fsproj to generate the scanner and parser.

In [ ]:
File.ReadAllText("GrammarParser/Ast.fs") |> printfn "%s"

`grammarToString renderRule grammar` renders a complete HTML document. Passing each next renderer explicitly lets the helpers appear in the original teaching order; they are composed once in `main`.

In [ ]:
let grammarToString (renderRule: Rule -> string) grammar =
    "<!doctype html><html><head><meta charset='utf-8'><title>Grammar</title></head><body><table>\n" +
    (grammar |> List.map renderRule |> String.concat "\n") +
    "</table></body></html>"

`ruleToString renderBody rule` emits table rows for all alternatives. The head has an HTML `id` anchor, and later alternatives are prefixed by `|`.

In [ ]:
let ruleToString (renderBody: Item list -> string) (rule: Rule) =
    let head = WebUtility.HtmlEncode rule.Head
    rule.Alternatives
    |> List.mapi (fun index body ->
        let title = if index = 0 then "<th scope='row' id='" + head + "'><em>" + head + "</em></th>" else "<td></td>"
        let separator = if index = 0 then ":" else "|"
        "<tr>" + title + "<td><code>" + separator + "</code></td><td>" + renderBody body + "</td></tr>")
    |> String.concat "\n"
    |> fun rows -> rows + "\n<tr><td></td><td><code>;</code></td><td></td></tr>"

`bodyToString renderItem body` renders a sequence of grammar items, using an explicit empty-production marker for an empty list.

In [ ]:
let bodyToString renderItem body =
    match body with
    | [] -> "<code>/* empty */</code>"
    | items -> items |> List.map renderItem |> String.concat " "

`itemToString item` links variables and escapes quoted literals as HTML text. Escaping is essential for operators such as `<`, `>` and `&`.

In [ ]:
let itemToString = function
    | Variable name ->
        let escaped = WebUtility.HtmlEncode name
        "<a href='#" + escaped + "'><em>" + escaped + "</em></a>"
    | Literal text -> "<code>" + WebUtility.HtmlEncode text + "</code>"

In [ ]:
let main () =
    let grammar = Grammar.parse (File.ReadAllText "c-grammar.g")
    let renderBody = bodyToString itemToString
    let renderRule = ruleToString renderBody
    let html = grammarToString renderRule grammar
    File.WriteAllText("c-grammar.html",html)
    grammar,html

In [ ]:
let grammar,html = main ()
printfn "Rendered %d rules to c-grammar.html" grammar.Length
checkNotebook (grammar.Length > 50)
checkNotebook (html.Contains("id='primary_expression'") && html.Contains("href='#expression'"))
checkNotebook ((itemToString (Literal "'<'")).Contains("&lt;"))
let epsilon = Grammar.parse "start: | 'a';"
checkNotebook (epsilon.Head.Alternatives.Head.IsEmpty)

In [ ]:
// Open c-grammar.html from the notebook file browser.

The generated F# sources and build products are excluded by `.gitignore`; no broad file deletion is necessary.

In [ ]:
// Build products are ignored; authored grammar specifications are preserved.

In [ ]:
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(html), "text/html") |> ignore